# Qualitative results: ligand & pocket

Rows represent pockets; columns show **Reference → VoxBind → VoxBind + Ours → additional baselines**. The molecular meshes, colors, lighting, and default camera are taken directly from `fig-overview.ipynb` (or legacy `fig1.ipynb`). Protein-pocket atom occupancy is displayed; no density or voxel lattice is shown.

For the IDE-friendly standalone viewer, run `bash notebook/figures/run_fig_qual.sh` from the repository root and open the printed local URL. The remaining notebook cells are retained for reproducible static export.

1. Choose methods and pockets below, then run the cells in order.
2. Use each row's pocket selector and each method's ligand selector. Indices are **zero-based record indices in the original SDF**; individual SDF filenames are also shown.
3. Rotate or zoom any panel to **synchronize the camera across its row**. Click **Inspect selected ligands** to see 2D structures, SMILES, and source paths.
4. Run the last cell to save the current selection and cameras as PNG, PDF, SVG, and a reproducibility JSON.

Default pockets are the first two IDs with SDF files for every selected method (or the one available pocket for a single-pocket result bundle). For methods with cached per-molecule scores, the default ligand has the lowest verified `ENERGY_KEY` score among RDKit-readable samples. The default is `vina_dock`. This selects a molecule by its cached docking score; the displayed coordinates remain its original generated SDF pose. Methods without verified scores retain the first readable ligand and are explicitly marked as unscored. Manual ligand selection remains available.

Scores are matched to unique molecular identities, because evaluation indices may refer to a filtered list rather than raw SDF records. Ambiguous duplicate structures are left unscored. The score, original SDF index, evaluation index, and score-file path are included in the selection report and export JSON. These are selected examples, not representative averages.


## Kernel / dependencies

On this server, use the `voxbind` Python 3.12 kernel with the prepared `.cache/fig-qual` runtime. In another environment, use a Jupyter kernel with RDKit installed. If necessary, run the following commands **in that kernel**, then restart it.

```python
%pip install "plotly>=6.1.1" ipywidgets anywidget "kaleido>=1" nbformat pillow scikit-image
# If RDKit is missing: %pip install rdkit
# Only if Kaleido cannot find Chrome:
# import plotly.io as pio
# pio.get_chrome()
```

Plotly 3D scenes use WebGL. Molecular scenes are embedded as raster images in PDF/SVG, while titles and labels remain vector text. Set `EXPORT_SCALE` in the last cell to control resolution.


In [1]:
from pathlib import Path
import sys

# Locate the setup module without depending on the overview notebook's filename.
FIGURE_DIR = next(
    (p / "notebook/figures" for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / "notebook/figures/qual_setup.py").is_file()), None
)
if FIGURE_DIR is None:
    raise FileNotFoundError("Start the kernel inside the VoxBind repository.")
if str(FIGURE_DIR) not in sys.path:
    sys.path.insert(0, str(FIGURE_DIR))
import importlib, qual_setup
globals().update(importlib.reload(qual_setup).initialize())

print(f"Initialized {len(catalog.specs)} methods; style: {STYLE_PATH.name}")
display(make_table(catalog.coverage()))


Initialized 9 methods; style: fig-overview.ipynb


HTML(value='<table style="border-collapse:collapse;text-align:left;font-size:12px"><tr><th style=\'padding:5px…

## Methods & pockets

`METHODS` sets the column order after Reference. When the original experiment directories are unavailable, **VoxBind** and **VoxBind + Ours** automatically use `results/task2-drugdesign/VoxBind-base-ep350/samples` and `results/task2-drugdesign/VoxBind-Ours/samples`, respectively. Additional SDF baselines on this server are TargetDiff, FuncBind, AR, and Pocket2Mol. Optional runs include `"VoxBind + Ours v2"` and `"VoxBind σ=1.0"`. FuncBind GPU shards are combined into one method. DecompDiff currently has raw sampling output but no SDF files, so its coverage is zero.

To register another method, add its `root` and `layout` to `METHOD_SPECS`, then recreate the catalog. Supported layouts: `eval` (`target_XX/samples.sdf`), `eval_shards` (`gpuN/samples/target_XX/samples.sdf`), `sweep` (`id_N/run/SDF/*.sdf`), and `mcp_results` (`mcpp_<pdb>_<run>/pooled_*.sdf`).

AR/Pocket2Mol use the server's verified common CrossDocked mapping, `id_N == target_N` (see `base_drug/export_targetdiff_sdf.py`). If multiple runs exist, the last run name containing SDF files is used; runs are never pooled. Exact source paths are available through Inspect and the saved JSON. Reference and pocket files come from Ours v1. Other eval-format methods are checked for matching reference identity, reference coordinates, and pocket coordinates.

`ENERGY_KEY` controls ranking; `vina_min` and `vina_dock` select molecules using the corresponding cached evaluation scores but still display their original SDF coordinates. No docking or energy calculation is performed by this notebook. The default score source is `eval_docking_results.json` in each method root; an explicit `energy_file` may be supplied in `METHOD_SPECS`. The bundled VoxBind + Ours path explicitly uses its primary `eval_docking_results_full79.json`; the bundled base model retains its original per-target metric files and is shown as unscored by this selector.


In [2]:
# Allow this settings cell to run directly after a kernel restart.
if "catalog" not in globals():
    from pathlib import Path
    import sys

    # Locate the setup module without depending on the overview notebook's filename.
    FIGURE_DIR = next(
        (p / "notebook/figures" for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
         if (p / "notebook/figures/qual_setup.py").is_file()), None
    )
    if FIGURE_DIR is None:
        raise FileNotFoundError("Start the kernel inside the VoxBind repository.")
    if str(FIGURE_DIR) not in sys.path:
        sys.path.insert(0, str(FIGURE_DIR))
    import importlib, qual_setup
    globals().update(importlib.reload(qual_setup).initialize())

# Default comparison: Reference, downloaded VoxBind base ep350, and VoxBind + Ours.
METHODS = ["VoxBind", "VoxBind + Ours"]
# METHODS = ["VoxBind", "VoxBind + Ours", "TargetDiff", "FuncBind", "AR", "Pocket2Mol"]
ENERGY_KEY = "vina_dock"  # Alternatives: "vina_score", "vina_min". Lower is better.
# Example: METHODS += ["VoxBind + Ours v2"]

COMMON_TARGETS = catalog.common_targets(METHODS)
print(f"Common pockets ({len(COMMON_TARGETS)}): {', '.join(COMMON_TARGETS)}")
if not COMMON_TARGETS:
    raise ValueError("At least one pocket with SDF files for every selected method is required. Check METHODS and paths.")

# Top best-vina_dock improvements among pockets with an open-cone half-angle >= 30°.
# Δ (Ours - VoxBind): -6.04, -5.92, -5.79, -5.63 kcal/mol, respectively.
PREFERRED_POCKETS = ["target_83", "target_79", "target_57", "target_92"]
POCKET_IDS = PREFERRED_POCKETS if all(t in COMMON_TARGETS for t in PREFERRED_POCKETS) else COMMON_TARGETS[:4]
PANEL_PX = 310
POCKET_HALF_EXTENT_A = 8.0  # Pocket heavy atoms in fig1's reference-centered 16 Å cube.

# Optional: restore methods, pockets, records, cameras, and panel size from a previous export.
RESTORE_JSON = None  # Example: FIGURE_DIR / "exports/fig-qual/fig-qual.selection.json"
RESTORED = json.loads(Path(RESTORE_JSON).read_text()) if RESTORE_JSON else {}
if RESTORED:
    METHODS = RESTORED["methods"]
    ENERGY_KEY = RESTORED.get("energy_key", "vina_score")
    POCKET_IDS = RESTORED["targets"]
    PANEL_PX = RESTORED["panel_px"]
    POCKET_HALF_EXTENT_A = RESTORED["pocket_half_extent_A"]
missing = [t for t in POCKET_IDS if t not in COMMON_TARGETS]
if missing:
    raise ValueError(f"Pinned pockets not available for every method in METHODS: {missing}")
print("Rows:", POCKET_IDS)
print("Columns:", ["Reference", *METHODS])


Common pockets (79): target_02, target_03, target_04, target_05, target_07, target_08, target_09, target_10, target_11, target_12, target_14, target_15, target_16, target_17, target_18, target_19, target_22, target_23, target_24, target_25, target_27, target_28, target_29, target_30, target_31, target_33, target_34, target_36, target_37, target_38, target_39, target_40, target_41, target_43, target_44, target_45, target_46, target_47, target_48, target_49, target_50, target_51, target_55, target_56, target_57, target_58, target_60, target_62, target_63, target_64, target_65, target_66, target_68, target_69, target_70, target_71, target_72, target_73, target_74, target_76, target_77, target_78, target_79, target_80, target_81, target_82, target_83, target_85, target_87, target_88, target_89, target_90, target_91, target_92, target_93, target_95, target_96, target_98, target_99
Rows: ['target_85', 'target_83']
Columns: ['Reference', 'VoxBind', 'VoxBind + Ours']


## Best docking energy per method

Lowest cached `ENERGY_KEY` (`vina_dock` by default; kcal/mol, lower is better) among each method's scored molecules, for the two selected pockets — a quick summary before the interactive viewer. This reuses the same cached docking scores the selector uses; **no docking is run here**. Each cell shows the best value and `(scored / total molecules)`; methods without a verified score file show `unscored`.


In [3]:
# Best cached docking energy per method for the selected pockets (no docking is run here).
_energy_rows = []
# Crystal reference ligand first, then each method's best generated molecule.
_ref_row = {"method": "Reference (crystal)"}
for _target in POCKET_IDS:
    _re = catalog.reference_energy(_target, ENERGY_KEY)
    _ref_row[_target] = f"{_re:.3f}" if _re is not None else "n/a"
_energy_rows.append(_ref_row)
for _method in METHODS:
    _row = {"method": _method}
    for _target in POCKET_IDS:
        _recs, _ = catalog.ranked_records(_method, _target, ENERGY_KEY)
        _es = [r["energy"] for r in _recs if r.get("energy") is not None]
        _row[_target] = f"{min(_es):.3f}  ({len(_es)}/{len(_recs)})" if _es else "unscored"
    _energy_rows.append(_row)
print(f"Best {ENERGY_KEY} per method (kcal/mol, lower is better) · method cell = best (scored/total); Reference = crystal ligand")
display(make_table(_energy_rows))


Best vina_dock per method (kcal/mol, lower is better) · method cell = best (scored/total); Reference = crystal ligand


HTML(value='<table style="border-collapse:collapse;text-align:left;font-size:12px"><tr><th style=\'padding:5px…

## Interactive comparison

Ligand: `#F5B27E` · Pocket: `#8291E8` · pocket atom occupancy: `#B4BEF0` (opacity `0.1`) · 16 Å data-cube edges: `#514F52` · ball/stick radius: `0.32 Å` · white background. Column labels live in the controls above the visualizer and are omitted inside the 3D panes.

The reference ligand's heavy-atom centroid is subtracted **identically from every ligand and the pocket**. Generated ligands are never individually aligned or recentered. Original SDF poses are shown without redocking. The pocket crop stays fixed within each row; ligands are never cropped. Selecting a large ligand expands the axis range equally across the row.

Startup loads one selected 3D record per method. After the one-time score index is built, full RDKit validation and the complete ligand menu run only when **Load all ligands** is clicked; the skipped count is then shown. Scroll horizontally if necessary. Changing a ligand updates only its column's molecular meshes and retains the row's camera. Pocket meshes are reused, and uniform role colors are sent once per mesh. Inspect computes 2D depictions on copies, preserving the original 3D coordinates.

Score-to-SDF mappings are cached under `.cache/fig-qual/selection-index`. If source files or the RDKit version change, the affected index is rebuilt once. The cache stores record identities and scores, not a new docking calculation. Ordinary loading reads only the selected molecule. Ligand changes update existing WebGL scenes without removing and redisplaying the row.


In [4]:
# Rerunning this cell rebuilds the UI from the settings above.
if "grid" in globals():
    if hasattr(grid, "close"):
        grid.close()
    else:
        # Compatibility with an instance created before the cleanup method was added.
        for old_row in grid.rows:
            old_row["figure"].close()
        grid.widget.close()
print(f"Loading one selected ligand per method for {len(POCKET_IDS)} pockets...", flush=True)
grid = QualitativeGrid(
    catalog, STYLE, METHODS, POCKET_IDS,
    panel_px=PANEL_PX, pocket_extent=POCKET_HALF_EXTENT_A,
    selections=RESTORED.get("selections"), cameras=RESTORED.get("cameras"),
    energy_key=ENERGY_KEY,
)
display(grid.widget)
print("Python loading complete. The interactive widget is now rendered by the browser.", flush=True)


Loading one selected ligand per method for 2 pockets...


Python loading complete. The interactive widget is now rendered by the browser.


## Check the current selection

Inspect the original SDF paths and record indices selected in the UI. `centroid_offset_A` is the distance from the reference centroid; offsets above 10 Å are flagged for inspection. This distance is neither an alignment metric nor a docking score. Rerun this cell after changing the selection to refresh the table.


In [5]:
display(make_table(grid.selection_report()))
# Inspect the current reproducibility settings in memory.
selection = grid.manifest()


HTML(value='<table style="border-collapse:collapse;text-align:left;font-size:12px"><tr><th style=\'padding:5px…

## Export — run this cell last

The current **pockets, per-method ligands, rotation, and zoom** are read directly from the live UI. You do not need to rerun the selection-report cell. Existing exports with the same name are overwritten; change `OUTPUT_STEM` to keep another version.


In [6]:
OUTPUT_DIR = FIGURE_DIR / "exports/fig-qual"
OUTPUT_STEM = "fig-qual"
EXPORT_SCALE = 3  # Resolution multiplier for PNG and embedded WebGL scenes.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

final_fig = grid.figure()  # Snapshot the current live selection and cameras.
export_paths = [OUTPUT_DIR / f"{OUTPUT_STEM}.{ext}" for ext in ("png", "pdf", "svg")]
try:
    pio.write_images(
        fig=[final_fig] * len(export_paths),
        file=[str(p) for p in export_paths],
        format=["png", "pdf", "svg"], scale=EXPORT_SCALE,
        width=final_fig.layout.width, height=final_fig.layout.height,
    )
except Exception as exc:
    raise RuntimeError(
        "Image export failed. This kernel needs recent plotly/kaleido, Chrome, "
        "and Chrome's shared libraries. See the dependency instructions above. Cause: " + str(exc)
    ) from exc

manifest_path = OUTPUT_DIR / f"{OUTPUT_STEM}.selection.json"
manifest_path.write_text(json.dumps(grid.manifest(), indent=2, ensure_ascii=False) + "\n")
for path in [*export_paths, manifest_path]:
    print(f"Saved: {path} ({path.stat().st_size:,} bytes)")


RuntimeError: Image export failed. This kernel needs recent plotly/kaleido, Chrome, and Chrome's shared libraries. See the dependency instructions above. Cause: 

Kaleido requires Google Chrome to be installed.

Either download and install Chrome yourself following Google's instructions for your operating system,
or install it from your terminal by running:

    $ plotly_get_chrome

